# README

- **Author**: RUEI-HSIANG, CHANG（張睿庠，B12303030；AI協助）
- **Created At**: 2026-11-09
- **Last Modified At**: 2026-11-09

---

## What does this file do?

針對2026-11-08全國研究母表中1,223位缺值候選人，套用本輪逐來源已讀完文字的人工判讀，只補原先為空的G5／G6。保留全部已知值、G1–G4、資格規則、游智彬暫納入與兩市210人版本。原始搜尋已涵蓋1,221個G6缺值者各兩式，以及346個G5缺值者各一式；**搜得／擷取不等於已判讀**。未審來源或未解身分不自動填0。

G5為第一次參選任何民選公職；本人曾落選或任里長、代表亦為0。三筆1依完整報導的素人／首度參選敘事與身分線索，保留Yang修訂規則。G6採配偶與三親等內曾任民選或政治任命公職；一般軍職、常任文官、單純黨職／幕僚或落選親屬不單獨計入。已完成指定搜尋且全文未見合格親屬者可依學生規則編「搜尋未發現」0，絕非證明不存在。

---

## What does this file take?

- **Source Data Sets**：下列`PINS`逐檔列明精確路徑及SHA-256；來源文件為[data/raw/national_gap_restore_20261109_documentation.ipynb](../../data/raw/national_gap_restore_20261109_documentation.ipynb)。2026-11-08研究母表、缺口、兩市210人版本及本輪搜尋／完整擷取文字／人工判讀輸入均在讀取前驗證。董昌華另讀內政部既存原始HTML作職涯身分連結。
- 原始採集由Codex網路搜尋及requests執行，已保存回應與文字；本Notebook重現**判讀後回填與驗證**，不宣稱以程式重做人工全文閱讀。

---

## What does this file output?

`data/processed/national_gap_restore_{roster,all,mayor,council,adjudications,candidate_progress,gap_queue,coverage}_20261109.csv`八個具名輸出（實際程式字典逐檔列明），及同prefix的manifest／validation JSON。角色依序為登記稽核母表、三池含缺值研究輸入、逐格證據稽核、1,223人進度、剩餘缺口及22縣市覆蓋表。文件：[data/processed/national_gap_restore_20261109_documentation.ipynb](../../data/processed/national_gap_restore_20261109_documentation.ipynb)。尚未完成全文判讀的候選人保留缺值。本輪**不重跑分群**，舊351人分析仍讀原11/08檔案。

全部輸出均存入`data/processed`、與輸入不同路徑，儲存後逐格回讀核對、驗證schema／鍵／三池與資格／已知值保存，另計算實得SHA-256。


In [1]:
from pathlib import Path
import json, hashlib, re
import pandas as pd
from IPython.display import display
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
            if (p/'AGENTS.md').is_file() and (p/'templates').is_dir())
def sha(p): return hashlib.sha256(p.read_bytes()).hexdigest()
def read_jsonl(p): return [json.loads(line) for line in p.read_text().splitlines() if line.strip()]


In [2]:
PINS = {'data/processed/national_g5_g6_roster_20261108.csv': 'b3e063993ee0c26d9513cc175ebfb03837dce90397f11585303d459512c7b944', 'data/processed/national_g5_g6_all_20261108.csv': '961a7c075044cc66fba259fad930b0c0963935d5c8da6f754a93e8fdbd808632', 'data/processed/national_g5_g6_gap_queue_20261108.csv': '412bf1f0f0c38d90c9d13b4b9faaeb2076630e4774186b96cafbdf30e94b527c', 'data/processed/taipei_kaohsiung_yu_provisional_all_20261108.csv': '35e3b85e407d4eb91e5c96043d06a407d21e070a5920032b26f74e916f883cbe', 'data/raw/national_gap_restore_20261108/G5_profile_source_leads.json': '81d3288ae6a716b278cb91cd1f1b185619890dae8a8bae16995636a26c98dd26', 'data/raw/national_gap_restore_20261108/alias_query_plan.json': '7c5f51949ab6381a115deb42920df66c394b195d611a18c8389b6f820108b8d3', 'data/raw/national_gap_restore_20261108/article_capture_manifest.json': 'e6b6f0d97b2fe6f92498acc4337bef2e6774ca8c043716f614978e13d3f80a2d', 'data/raw/national_gap_restore_20261108/collection_checkpoint.json': '37ed1c2288eaaeebaad546f0e780c947dcf2020d205a7dbe85e5b0d75e8ddbe8', 'data/raw/national_gap_restore_20261108/county_election_articles.json': 'bfa870d191c9887d12e3c704904d95acd77b7341a5aa9fbb27e17c43aea6cd33', 'data/raw/national_gap_restore_20261108/followup_leads.json': 'e6a7565825c1f570f56d372266c8b7e69c5fb636a7dd0642dddfa1b8e9ef3e11', 'data/raw/national_gap_restore_20261108/g5_followup_query_plan.json': '3e062f80e0698b99f82c2dda5b8e29f9367f3fe3b88e47ff5ef83ae765695956', 'data/raw/national_gap_restore_20261108/identity_and_scope_review_notes.json': 'fd3c26642cea8aa2f30c178122c2f9ecdf01a2a3f2d91332d1ade3682d1e1d16', 'data/raw/national_gap_restore_20261108/query_plan.json': '81643739e1f014604227654f45071a3edb566fd353f453ef7130bcd06f515c5f', 'data/raw/national_gap_restore_20261108/reviewed_adjudications.jsonl': '520a17aa37ffdeed6786dc1fffe62c118cc5a7872a09994b8ebe61a48620d445', 'data/raw/national_gap_restore_20261108/search_responses.jsonl': 'ac561219ed9c51fbf51c44cd79d63e41955ee75734b2e7d2bf6fdd49601e7e0f', 'data/raw/national_gap_restore_20261108/source_bodies.jsonl': 'b8908403ebe6ce4f5a970bcdc8a1bbde10ac0348cc96335bc800c965fac7ec0a', 'data/raw/inventory_20261019/ER11309VB00001.html': '1d547fa37f4307649f92e4ff612a948d4d8fc0cd2fc162450d8f87041edc5023'}
for rel, expected in PINS.items():
    assert sha(ROOT/rel) == expected, f'Changed input: {rel}'
RAW = ROOT/'data/raw/national_gap_restore_20261108'
KEY='candidate_key'; G5='G5_first_time_any_elected_office'; G6='G6_political_family'
base = pd.read_csv(ROOT/'data/processed/national_g5_g6_roster_20261108.csv',dtype=str,keep_default_na=False)
old = pd.read_csv(ROOT/'data/processed/national_g5_g6_all_20261108.csv',dtype=str,keep_default_na=False)
gap_before = pd.read_csv(ROOT/'data/processed/national_g5_g6_gap_queue_20261108.csv',dtype=str,keep_default_na=False)
assert base.shape == (1583,56) and len(old)==1574 and len(gap_before)==1223
assert base[KEY].is_unique and old[KEY].is_unique and gap_before[KEY].is_unique
assert set(gap_before[KEY]) == set(old.loc[old[[G5,G6]].eq('').any(axis=1),KEY])
search=read_jsonl(RAW/'search_responses.jsonl'); articles=read_jsonl(RAW/'source_bodies.jsonl')
decisions=read_jsonl(RAW/'reviewed_adjudications.jsonl')
by_article={a['id']:a for a in articles}; assert len(by_article)==len(articles)
query_rows=[dict(q,batch_start=b['batch_start'],retrieval_date=b['retrieval_date']) for b in search for q in b['queries']]
queries=pd.DataFrame(query_rows)
assert len(search)==699 and len(queries)==2792
assert queries.groupby(['candidate_key','round']).size().eq(1).all()
assert queries[queries['round'].isin([1,2])].groupby(KEY).size().eq(2).all()
assert queries[queries['round'].isin([1,2])][KEY].nunique()==1221
assert set(queries.loc[queries['round'].isin([1,2]),KEY])==set(gap_before.loc[gap_before.missing_G6.eq('1'),KEY])
assert set(queries.loc[queries['round'].eq(3),KEY])==set(gap_before.loc[gap_before.missing_G5.eq('1'),KEY])
for b in search: assert hashlib.sha256(b['response'].encode()).hexdigest()==b['response_sha256']
for a in articles:
    if 'text' in a: assert hashlib.sha256(a['text'].encode()).hexdigest()==a['body_sha256']
assert len({(x[KEY],x['variable']) for x in decisions})==len(decisions)
base_ix=base.set_index(KEY); norm=lambda s:re.sub(r'\s+','',s)
for x in decisions:
    assert x[KEY] in set(gap_before[KEY]) and x['variable'] in [G5,G6] and x['value'] in ['0','1']
    assert base_ix.at[x[KEY],x['variable']]=='', 'Only original gaps may be filled'
    a=by_article[x['article_id']]
    assert 'text' in a and a['url']==x['source_url'] and a['body_sha256']==x['body_sha256']
    assert x['full_article_body_read'] is True
    if x.get('supporting_excerpt'): assert norm(x['supporting_excerpt']) in norm(a['text'])
    if x['variable']==G6 and x['value']=='1': assert x.get('supporting_excerpt')
    if x['variable']==G6 and x['value']=='0':
        assert len(queries[(queries[KEY]==x[KEY]) & queries['round'].isin([1,2])])==2
        assert 'nonfinding' in x['quality']
    for rel, expected in x.get('supporting_files',{}).items(): assert PINS[rel]==expected
print(f'PASS: {len(search)} search batches, {len(queries)} queries, {len(articles)} source captures, {len(decisions)} unique reviewed cells.')


PASS: 699 search batches, 2792 queries, 2250 source captures, 279 unique reviewed cells.


## Processing

只採已完成判讀的逐格決定；新日期狀態另欄保存，11/08日期欄仍是舊快照。

In [3]:
d=base.copy().set_index(KEY)
for variable, tag in [(G5,'G5'),(G6,'G6')]:
    d[f'{tag}_restoration_status_20261109']=d[variable].map(lambda v:'carried_forward' if v else 'pending_review')
    d[f'{tag}_restored_this_pass_20261109']='0'
for rno,tag in [(3,'G5'),(1,'G6')]:
    q=queries[queries['round'].eq(rno)].groupby(KEY).size()
    d[f'{tag}_search_queries_20261109']=d.index.map(q).fillna(0).astype(int).astype(str)
# G6 counts include both core searches and extra alias searches.
g6q=queries[queries['round'].isin([1,2,4])].groupby(KEY).size()
d['G6_search_queries_20261109']=d.index.map(g6q).fillna(0).astype(int).astype(str)
d['restoration_checked_date_20261109']=''
for x in decisions:
    key=x[KEY]; var=x['variable']; tag='G5' if var==G5 else 'G6'
    d.at[key,var]=x['value']
    d.at[key,f'{tag}_restoration_status_20261109']=x['quality']
    d.at[key,f'{tag}_restored_this_pass_20261109']='1'
    d.at[key,'restoration_checked_date_20261109']=x['checked_date']
    d.at[key,f'{tag}_source']=x['source_url']+'; data/raw/national_gap_restore_20261108/source_bodies.jsonl#'+x['article_id']
    d.at[key,f'{tag}_evidence']=x['evidence_note']
    if tag=='G5':
        d.at[key,'G5_label']='首次參選任何民選公職' if x['value']=='1' else '非首次參選任何民選公職'
        d.at[key,'G5_audit_status']=x['quality']
    else:
        d.at[key,'G6_label']='有政治家族' if x['value']=='1' else '搜尋未發現符合政治家族'
        d.at[key,'G6_evidence_tier']=x['quality']
# Historical dated fields remain snapshots of Nov 8; use Nov 9 status columns for this pass.
for var in [G5,G6]:
    known=base_ix[var].ne('');assert d.loc[known,var].equals(base_ix.loc[known,var])
allowed_changed={G5,G6,'G5_label','G5_source','G5_evidence','G5_audit_status','G6_label','G6_source','G6_evidence','G6_evidence_tier'}
for c in base_ix:
    if c not in allowed_changed: assert d[c].equals(base_ix[c]),c
roster=d.reset_index(); all_pool=roster[roster.national_research_included_20261108.eq('1')].reset_index(drop=True)
mayor=all_pool[all_pool.office_type.isin(['直轄市長','縣（市）長'])].reset_index(drop=True)
council=all_pool[all_pool.office_type.isin(['直轄市議員','縣（市）議員'])].reset_index(drop=True)
assert (len(all_pool),len(mayor),len(council))==(1574,81,1493)
assert set(mayor[KEY]).isdisjoint(council[KEY]) and set(mayor[KEY])|set(council[KEY])==set(all_pool[KEY])
assert set(all_pool[KEY])==set(old[KEY])
assert all_pool.loc[all_pool.candidate_name.eq('游智彬'),'eligibility_provisional_flag_20261108'].tolist()==['1']
tk=pd.read_csv(ROOT/'data/processed/taipei_kaohsiung_yu_provisional_all_20261108.csv',dtype=str,keep_default_na=False).set_index(KEY)
# No targets are in Taipei/Kaohsiung; every original TK cell remains the same as the baseline.
old_ix=old.set_index(KEY);new_ix=all_pool.set_index(KEY)
assert new_ix.loc[tk.index,old_ix.columns].equals(old_ix.loc[tk.index])
progress=gap_before[[KEY,'candidate_name','county_city','office_type',G5,G6]].rename(columns={G5:'G5_before',G6:'G6_before'}).merge(
    all_pool[[KEY,G5,G6,'G5_restoration_status_20261109','G6_restoration_status_20261109','G5_search_queries_20261109','G6_search_queries_20261109']],on=KEY,validate='one_to_one')
progress['G5_restored']=((progress.G5_before=='') & progress[G5].ne('')).astype(int)
progress['G6_restored']=((progress.G6_before=='') & progress[G6].ne('')).astype(int)
progress['complete_after']=progress[[G5,G6]].isin(['0','1']).all(axis=1).astype(int)
progress['review_status']=progress.complete_after.map({1:'both_features_complete',0:'still_missing'})
notes=json.loads((RAW/'identity_and_scope_review_notes.json').read_text())
notes_by={(x['county_city'],x['candidate_name'],x['variable']):x['reason'] for x in notes}
progress['specific_followup_note']=progress.apply(lambda x:'; '.join(v for (county,name,var),v in notes_by.items() if county==x.county_city and name==x.candidate_name and var in [G5,G6]),axis=1)
gap=progress[progress.complete_after.eq(0)].copy();gap['missing_G5']=gap[G5].eq('').astype(int);gap['missing_G6']=gap[G6].eq('').astype(int)
# Preserve each adjudication as a cell-level audit. Dictionary-valued auxiliary pins are serialized as JSON.
audit=pd.DataFrame(decisions)
if 'supporting_files' in audit:audit['supporting_files']=audit.supporting_files.map(lambda x:json.dumps(x,ensure_ascii=False,sort_keys=True) if isinstance(x,dict) else '')
audit=audit.fillna('')
coverage=[]
for pool,q in [('all',all_pool),('mayor',mayor),('council',council)]:
    for county,t in q.groupby('county_city',sort=True):
        valid=t[[G5,G6]].isin(['0','1']).all(axis=1)
        coverage.append({'pool':pool,'county_city':county,'roster_n':len(t),'complete_n':int(valid.sum()),'missing_candidate_n':int((~valid).sum()),'missing_G5_n':int(t[G5].eq('').sum()),'missing_G6_n':int(t[G6].eq('').sum()),'new_G5_cells':int(t.G5_restored_this_pass_20261109.eq('1').sum()),'new_G6_cells':int(t.G6_restored_this_pass_20261109.eq('1').sum()),'retention_percent':100*valid.mean()})
coverage=pd.DataFrame(coverage);assert len(coverage)==66
assert int(progress.G5_restored.sum()+progress.G6_restored.sum())==len(audit)
print('Newly resolved cells by variable/value:');display(audit.groupby(['variable','value']).size().rename('N').reset_index())
display(coverage[coverage.pool.eq('all')])


Newly resolved cells by variable/value:


,variable,value,N
0,G5_first_time_any_elected_office,0,92
1,G5_first_time_any_elected_office,1,3
2,G6_political_family,0,107
3,G6_political_family,1,77


,pool,county_city,roster_n,complete_n,missing_candidate_n,missing_G5_n,missing_G6_n,new_G5_cells,new_G6_cells,retention_percent
0,all,南投縣,63,14,49,13,45,8,13,22.222222
1,all,嘉義市,42,9,33,7,30,2,5,21.428571
2,all,嘉義縣,54,8,46,14,46,3,2,14.814815
3,all,基隆市,62,25,37,10,37,0,18,40.322581
4,all,宜蘭縣,64,7,57,13,57,3,1,10.937500
5,all,屏東縣,89,11,78,23,73,10,11,12.359551
6,all,彰化縣,90,21,69,12,68,9,9,23.333333
7,all,新北市,114,74,40,14,33,7,69,64.912281
8,all,新竹市,59,7,52,7,48,2,5,11.864407
9,all,新竹縣,71,9,62,19,62,4,3,12.676056


## Save and verify

回讀已儲存檔案，不把記憶體計算當作輸出驗證。

In [4]:
OUT=ROOT/'data/processed';prefix='national_gap_restore_';date='20261109'
outputs={prefix+'roster_'+date+'.csv':roster,prefix+'all_'+date+'.csv':all_pool,prefix+'mayor_'+date+'.csv':mayor,prefix+'council_'+date+'.csv':council,prefix+'adjudications_'+date+'.csv':audit,prefix+'candidate_progress_'+date+'.csv':progress,prefix+'gap_queue_'+date+'.csv':gap,prefix+'coverage_'+date+'.csv':coverage}
saved=[]
for name,frame in outputs.items():
    p=OUT/name;assert p.parent==OUT and p.name.endswith('_20261109.csv')
    assert str(p.relative_to(ROOT)) not in PINS
    frame.to_csv(p,index=False,encoding='utf-8-sig',lineterminator='\n')
    actual=pd.read_csv(p,dtype=str,keep_default_na=False)
    pd.testing.assert_frame_equal(actual,frame.fillna('').astype(str).reset_index(drop=True),check_dtype=False)
    assert actual[KEY].is_unique if KEY in actual and frame is not audit else True
    saved.append({'path':str(p.relative_to(ROOT)),'rows':len(actual),'columns':list(actual),'sha256':sha(p),'verification':'serialized_reopened_all_cells_equal'})
validation={'as_of':'2026-11-09','input_pins':PINS,'search_batches':len(search),'search_query_n':len(queries),'G6_candidates_two_core_searches':1221,'G5_candidates_followup_search':346,'captured_sources':sum('text' in a for a in articles),'failed_captures':sum('text' not in a for a in articles),'baseline_missing_candidates':1223,'baseline_missing_cells':1567,'resolved_cells':len(audit),'resolved_candidates':audit[KEY].nunique(),'resolved_G5_cells':int(audit.variable.eq(G5).sum()),'resolved_G6_cells':int(audit.variable.eq(G6).sum()),'G6_new_positive':int(((audit.variable==G6)&(audit.value=='1')).sum()),'G6_new_search_nonfinding':int(((audit.variable==G6)&(audit.value=='0')).sum()),'newly_complete_candidates':int(progress.complete_after.sum()),'remaining_missing_candidates':len(gap),'remaining_missing_G5':int(all_pool[G5].eq('').sum()),'remaining_missing_G6':int(all_pool[G6].eq('').sum()),'complete_case_n':{p:int(q[[G5,G6]].isin(['0','1']).all(axis=1).sum()) for p,q in [('all',all_pool),('mayor',mayor),('council',council)]},'research_roster_n':1574,'qualification_excluded_n':9,'TK_210_preserved':True,'Yu_provisional_inclusion_preserved':True,'known_values_preserved':True,'other_G1_to_G4_and_eligibility_fields_preserved':True,'missing_not_automatically_zero_filled':True,'clustering_rerun':False,'review_complete_for_all_collected_sources':False,'all_missing_data_restored':False,'serialized_outputs_reopened':True}
assert validation['remaining_missing_candidates']+validation['complete_case_n']['all']==1574
assert validation['newly_complete_candidates']==validation['complete_case_n']['all']-351
for name,obj in [(prefix+'manifest_'+date+'.json',saved),(prefix+'validation_'+date+'.json',validation)]:
    p=OUT/name;p.write_text(json.dumps(obj,ensure_ascii=False,indent=2));assert json.loads(p.read_text())==obj
for rel,expected in PINS.items():assert sha(ROOT/rel)==expected
print(json.dumps({k:v for k,v in validation.items() if k not in ['input_pins']},ensure_ascii=False,indent=2))
print('PASS: 8 CSV and 2 JSON outputs serialized and reopened; baseline inputs unchanged. No clustering executed.')


{
  "as_of": "2026-11-09",
  "search_batches": 699,
  "search_query_n": 2792,
  "G6_candidates_two_core_searches": 1221,
  "G5_candidates_followup_search": 346,
  "captured_sources": 2090,
  "failed_captures": 160,
  "baseline_missing_candidates": 1223,
  "baseline_missing_cells": 1567,
  "resolved_cells": 279,
  "resolved_candidates": 243,
  "resolved_G5_cells": 95,
  "resolved_G6_cells": 184,
  "G6_new_positive": 77,
  "G6_new_search_nonfinding": 107,
  "newly_complete_candidates": 149,
  "remaining_missing_candidates": 1074,
  "remaining_missing_G5": 251,
  "remaining_missing_G6": 1037,
  "complete_case_n": {
    "all": 500,
    "mayor": 80,
    "council": 420
  },
  "research_roster_n": 1574,
  "qualification_excluded_n": 9,
  "TK_210_preserved": true,
  "Yu_provisional_inclusion_preserved": true,
  "known_values_preserved": true,
  "other_G1_to_G4_and_eligibility_fields_preserved": true,
  "missing_not_automatically_zero_filled": true,
  "clustering_rerun": false,
  "review_comple